In [1]:
import pandas as pd
import numpy as np

# CSV dosyasını oku
df = pd.read_csv("../../data/hourly_observations.csv")

df["clothing_recommendation"].nunique()

8

In [2]:
df["clothing_recommendation"].value_counts()

clothing_recommendation
winter_coat_scarf_gloves             10180
light_jacket_or_sweater              10108
warm_jacket_layers                    8542
long_sleeves_light_layer              7966
heavy_winter_coat_gloves_hat          4648
t_shirt_comfortable                   1838
light_breathable_clothing              138
very_light_clothing_stay_hydrated       20
Name: count, dtype: int64

In [3]:
clothing_map = {#model bu sayede daha stabil olur, küçük class problemini azaltır
    "heavy_winter_coat_gloves_hat": "very_cold",
    "winter_coat_scarf_gloves": "very_cold",
    "warm_jacket_layers": "cold",
    "light_jacket_or_sweater": "cold",
    "long_sleeves_light_layer": "mild",
    "t_shirt_comfortable": "mild",
    "light_breathable_clothing": "hot",
    "very_light_clothing_stay_hydrated": "hot"
}

df["clothing_group"] = df["clothing_recommendation"].map(clothing_map)

In [4]:
features = [
  "temperature_c",
    "humidity_pct",
    "pressure_hpa",
    "wind_speed_kmh",
    "wind_gust_kmh",
    "cloud_cover_pct",
    "visibility_km",
    "uv_index",
    "hour_of_day",
    "month"
]


X = df[features]
#y = df["clothing_group"]   # önce grouping yaptıysan
y = df["clothing_recommendation"]  # grouping yapmadıysan bunu kullan



In [5]:
from sklearn.model_selection import train_test_split

#x_test for only testing data and x_train for only training data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42#tüm datanın %80'i traine harcanıyor gerisi test için, random_state ile bu bölmeyi rastgale yapıyoruz
)

X_train["temperature_c"] += np.random.normal(0, 2, size=len(X_train))#ezberlemeyi engellemek için noise ekliyoruz

In [6]:
from sklearn.ensemble import RandomForestClassifier

model_clothing = RandomForestClassifier(n_estimators=600, random_state=42, class_weight="balanced")#100 ağaç, her çalıştırmada accuracy değişmemesi için random_state atıyoruz(sabit rastgalelik)
model_clothing.fit(X_train, y_train)#ilişkiyi öğrenmeye çalışıyor, train kısmı

y_pred = model_clothing.predict(X_test)

In [7]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))#gerçek sonuç ile trainden sonraki tahmin edilen veriyi kıyasla, rapot üret

                                   precision    recall  f1-score   support

     heavy_winter_coat_gloves_hat       0.95      0.90      0.92       929
        light_breathable_clothing       0.75      0.12      0.20        26
          light_jacket_or_sweater       0.90      0.91      0.90      2066
         long_sleeves_light_layer       0.89      0.90      0.89      1581
              t_shirt_comfortable       0.85      0.80      0.83       358
very_light_clothing_stay_hydrated       0.00      0.00      0.00         6
               warm_jacket_layers       0.90      0.89      0.89      1693
         winter_coat_scarf_gloves       0.90      0.94      0.92      2029

                         accuracy                           0.90      8688
                        macro avg       0.77      0.68      0.70      8688
                     weighted avg       0.90      0.90      0.90      8688



c:\Users\ghost\OneDrive\Masaüstü\Weatherwise\venv_sklearn172\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\ghost\OneDrive\Masaüstü\Weatherwise\venv_sklearn172\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\ghost\OneDrive\Masaüstü\Weatherwise\venv_sklearn172\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavio

In [8]:
import pandas as pd

importance = model_clothing.feature_importances_

feature_importance = pd.DataFrame({
    "feature": X.columns,
    "importance": importance
}).sort_values(by="importance", ascending=False)

feature_importance

,feature,importance
0,temperature_c,0.408605
9,month,0.113271
7,uv_index,0.079626
4,wind_gust_kmh,0.072397
1,humidity_pct,0.069152
3,wind_speed_kmh,0.068856
5,cloud_cover_pct,0.051272
2,pressure_hpa,0.048160
8,hour_of_day,0.045048
6,visibility_km,0.043612


In [9]:
for i in range(5):
    sample = X_test.iloc[[i]]#test setinden 1 tane değer al
    true_label = y_test.iloc[i]
    pred = model_clothing.predict(sample)[0]

    print(f"Sample {i}")
    print("Prediction:", pred)
    print("Actual:", true_label)
    print("-----")

Sample 0
Prediction: light_jacket_or_sweater
Actual: light_jacket_or_sweater
-----
Sample 1
Prediction: long_sleeves_light_layer
Actual: long_sleeves_light_layer
-----
Sample 2
Prediction: warm_jacket_layers
Actual: winter_coat_scarf_gloves
-----
Sample 3
Prediction: winter_coat_scarf_gloves
Actual: winter_coat_scarf_gloves
-----
Sample 4
Prediction: t_shirt_comfortable
Actual: light_breathable_clothing
-----


In [10]:
def generate_clothing_advice(clothing_pred):
    return f"Recommended clothing: {clothing_pred}"

sample = X_test.iloc[[0]]
pred = model_clothing.predict(sample)[0]

print(generate_clothing_advice(pred))

Recommended clothing: light_jacket_or_sweater


In [11]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, cross_validate, StratifiedKFold
from sklearn.metrics import classification_report, accuracy_score, f1_score
from xgboost import XGBClassifier
import numpy as np
import pandas as pd
from sklearn.utils import resample

df = pd.read_csv("../../data/hourly_observations.csv")

#clothing için
features = [
    "temperature_c",
    "wind_speed_kmh",
    "wind_gust_kmh",
    "uv_index",
    "hour_of_day",
    "month",
]

X = df[features]
y = df["clothing_recommendation"]


label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)

# Sadece train üzerinde oversampling
train_df = X_train.copy()
train_df["target"] = y_train

min_threshold = 300
balanced_train_parts = []

for label in train_df["target"].unique():
    class_df = train_df[train_df["target"] == label]

    if len(class_df) < min_threshold:
        class_df = resample(
            class_df,
            replace=True,
            n_samples=min_threshold,
            random_state=42
        )

    balanced_train_parts.append(class_df)

train_balanced = pd.concat(balanced_train_parts).sample(frac=1, random_state=42)

X_train_balanced = train_balanced[features]
y_train_balanced = train_balanced["target"]


xgb_model = XGBClassifier(
    n_estimators=400,
    max_depth=4,
    learning_rate=0.08,
    subsample=0.8,
    colsample_bytree=0.7,
    min_child_weight=5,
    gamma=0.1,
    reg_lambda=2,
    reg_alpha=0.1,
    random_state=42,
    eval_metric="mlogloss"
)

# CV'yi de sadece train üzerinde yap
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_results = cross_validate(
    xgb_model,
    X_train_balanced,
    y_train_balanced,
    cv=skf,
    scoring=["accuracy", "f1_weighted", "f1_macro"],
    return_train_score=True
)

xgb_model.fit(X_train_balanced, y_train_balanced)
y_pred = xgb_model.predict(X_test)

print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))
print("Accuracy:", accuracy_score(y_test, y_pred))

#hangi feature ne kadar yüzde etkiliyor
feature_importance = pd.DataFrame({
    "feature": X.columns,
    "feature": features,    
    "importance": xgb_model.feature_importances_
}).sort_values(by="importance", ascending=False)

print(feature_importance)

#Cross validation metrics
print("CV Accuracy:", cv_results["test_accuracy"])
print("CV F1 weighted:", cv_results["test_f1_weighted"])
print("CV F1 macro:", cv_results["test_f1_macro"])




                                   precision    recall  f1-score   support

     heavy_winter_coat_gloves_hat       0.98      0.99      0.99       930
        light_breathable_clothing       0.72      0.64      0.68        28
          light_jacket_or_sweater       0.99      0.99      0.99      2021
         long_sleeves_light_layer       0.99      0.99      0.99      1593
              t_shirt_comfortable       0.98      0.99      0.98       368
very_light_clothing_stay_hydrated       0.40      0.50      0.44         4
               warm_jacket_layers       0.99      0.99      0.99      1708
         winter_coat_scarf_gloves       0.99      0.99      0.99      2036

                         accuracy                           0.99      8688
                        macro avg       0.88      0.89      0.88      8688
                     weighted avg       0.99      0.99      0.99      8688

Accuracy: 0.9886049723756906
          feature  importance
0   temperature_c    0.513786
5       

In [12]:
import joblib
joblib.dump(xgb_model, "../../models/clothing_model.pkl")
joblib.dump(label_encoder, "../../models/clothing_label_encoder.pkl")

['../../models/clothing_label_encoder.pkl']

In [13]:
sample ={
  "temperature_c": 18.0,
  "wind_speed_kmh": 15.0,
  "wind_gust_kmh": 22.0,
  "uv_index": 2.0,
  "hour_of_day": 14,
  "month": 4,
}
sample_df = pd.DataFrame([sample])
prediction = xgb_model.predict(sample_df)
print(prediction)

[3]
